# STRATA, livro de códigos aninhado: kernel dos níveis 0 e 1 e segundo 8B (Qwen3-8B-Base)

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*. O notebook é autocontido: os arquivos do STRATA (patch do llama.cpp, kernels, quantizador e teste da GPU) vão embutidos numa célula.

A pergunta (docs/09, seção "Kernel dos níveis 0 e 1 e segundo 8B"): o empate do livro aninhado no Llama-3.1-8B se repete em outra família de modelo, agora **no formato real** (tabelas int8 por tensor, bloco do IQ4_XS byte a byte), e o kernel novo roda tão rápido quanto o IQ4_XS e o Q5_K_S?

O notebook:
1. compila o llama.cpp (commit fixo) com CUDA e os tipos novos `STRATA_LIVRO0` (136 bytes por 256 pesos, 4,25 bits/peso) e `STRATA_LIVRO1` (168 bytes, 5,25);
2. confere a GPU contra a CPU e contra float64 em multiplicações de matriz (lotes de 1 a 512) de todos os tipos do STRATA, com o IQ4_XS de controle;
3. faz o BF16, a imatrix (wikitext-2 de *treino*), a curva dos k-quants com imatrix, o IQ4_XS e o Q8_0;
4. quantiza com o livro aninhado no formato real: um GGUF por nível com os tipos novos e a desquantização exata em F16;
5. mede a KL contra o BF16 (wikitext-2 de *teste*) no caminho F16 dos dois lados (quem decide a qualidade), a KL dos tipos novos lidos pelo kernel na GPU e na CPU, e a velocidade com o `llama-bench`.

**Régua (escrita antes de rodar, docs/09):**
- *qualidade*, bits por peso a mais que a fronteira k-quants com imatrix + IQ4_XS no mesmo KL: **sucesso** se os níveis 0 e 1 ficam a no máximo +0,05; **zona cinza** se algum fica entre +0,05 e +0,10; **fracasso** acima de +0,10. Dominância à parte.
- *kernel, conferência*: a KL do tipo novo lido pelo kernel na GPU a 1 erro padrão da KL da emulação em F16; a CPU igual à GPU nos trechos conferidos.
- *kernel, velocidade de geração na GPU* (tg128): nível 0 ≥ 0,95× o IQ4_XS; nível 1 ≥ 0,95× o Q5_K_S.

Ocupa ~130 GB de disco no pico. Leva ~3 h na A100.

In [ ]:
KLD_CHUNKS = 100           # trechos na divergência KL (~78 MB de disco por trecho com o vocabulário do Qwen3)
CONF_CHUNKS = 5            # trechos da conferência CPU contra GPU (a CPU é lenta num 8B)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
RODADAS = 3                # rodadas alternadas do llama-bench
MODELO_HF = "Qwen/Qwen3-8B-Base"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100.'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 140:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor.')
if RAM < 50:
    print('Aviso: pouca RAM. Ligue a RAM alta.')

Os arquivos do STRATA (commit `cdaeae6` da branch `claude/project-thread-9jvckq`, mais as mudanças desta etapa): `kernel/` (referência em C, AVX2, CUDA e o patch do llama.cpp), `tests/kernel/strata_livro.py` (o quantizador) e `tests/kernel/strata_cuda_check.c` (GPU contra CPU). SHA-256 do pacote: `69e67dd484adf62a6a067b75ba93a2a502599d5916b858c9112fd47023d740bd`.

In [ ]:
import base64, io, tarfile, hashlib
PACOTE = "..."
raw = base64.b64decode(PACOTE)
assert hashlib.sha256(raw).hexdigest() == "69e67dd484adf62a6a067b75ba93a2a502599d5916b858c9112fd47023d740bd", 'pacote corrompido'
shutil.rmtree('/content/strata', ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall('/content/strata')
!find /content/strata -type f | sort

In [ ]:
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh /content/strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix llama-bench 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix|bench)$|libggml-cuda'

Conferência do CUDA contra a CPU: multiplica matrizes de cada tipo (Q5_K, Q6_K e IQ4_XS de controle, e os do STRATA, com os dois do livro e uma tabela qualquer) por lotes de 1, 2, 5, 8, 9, 64 e 512 colunas, nos dois backends, e compara os dois com uma referência em float64. Lotes de até 8 usam o produto escalar da geração; acima disso, a desquantização + cuBLAS. Tudo tem de sair OK.

In [ ]:
LIB = '/content/llama.cpp/build/bin'
!cc -O2 /content/strata/tests/kernel/strata_cuda_check.c -I llama.cpp/ggml/include -L {LIB} -lggml -lggml-base -lggml-cpu -lggml-cuda -lm -Wl,-rpath,{LIB} -o strata_cuda_check
out = !./strata_cuda_check 2>&1
print('\n'.join(out))
assert any(l.strip() == 'tudo OK' for l in out), 'o CUDA deu resultado diferente da CPU: avise na thread (cole a saída acima)'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Modelo em BF16, imatrix, k-quants e IQ4_XS com imatrix, Q8_0 e as duas bases da KL (100 trechos para a qualidade, 5 para a conferência CPU contra GPU). Os pesos originais do Hugging Face são apagados depois da conversão.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.txt"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q3_K_M', 'Q4_K_S', 'Q4_K_M', 'Q5_K_S', 'Q5_K_M', 'Q6_K', 'IQ4_XS']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
# um ponto entre o Q6_K e o Q8_0 (a curva tem 2 bits de buraco ali): Q6_K com as matrizes de atenção e a ffn_down em Q8_0
Q8MIX = ' '.join(f'--tensor-type {t}=q8_0' for t in ['attn_q', 'attn_k', 'attn_v', 'attn_output', 'ffn_down'])
!llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {Q8MIX} {G}/BF16.gguf {G}/Q6_K_Q8mix_imat.gguf Q6_K > {G}/Q6_K_Q8mix_imat.log 2>&1
!llama.cpp/build/bin/llama-quantize {G}/BF16.gguf {G}/Q8_0.gguf Q8_0 > {G}/Q8_0.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {CONF_CHUNKS} --kl-divergence-base /content/kld_conf.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin /content/kld_conf.bin
!df -h /content | tail -1

Funções de medida. `bpw` guarda os bits por peso nas matrizes das camadas (bytes reais para os k-quants; códigos + escalas + tabelas para o livro). `kl_de` aceita outra base e opções extras: `-ub 8` força o caminho da geração na GPU (produto escalar com ativações em q8_1); `-dev none -ngl 0 --no-op-offload` roda tudo na CPU.

In [ ]:
kld, bpw, erro, vel = {}, {}, {}, {}
LIVRO = f'python /content/strata/tests/kernel/strata_livro.py --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf'
GPU_ARGS, CPU_ARGS = '-ngl 99', f'-dev none -ngl 0 --no-op-offload -t {N}'

def kl_de(path, nome, base='/content/kld_base.bin', extra=GPU_ARGS):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base {base} --kl-divergence {extra} 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.3f} ± {1000 * kld[nome][1]:.3f}, mesmo 1º token {kld[nome][2]}%, {bpw.get(nome, float("nan")):.3f} bits/peso')

def bench(path):
    # tok/s de geração (tg128) e de prompt (pp512) na GPU, 3 repetições
    out = subprocess.run(['llama.cpp/build/bin/llama-bench', '-m', path, '-ngl', '99', '-p', '512', '-n', '128', '-r', '3', '-o', 'json'],
                         capture_output=True, text=True)
    try:
        return {('pp' if x['n_prompt'] else 'tg'): x['avg_ts'] for x in json.loads(out.stdout)}
    except Exception as e:
        print('Erro no llama-bench de', path, e, out.stderr[-500:])
        return {}

O livro aninhado no formato real (`--formato`): níveis 0 e 1, tabelas int8 por tensor. Sai um GGUF por nível com os tipos novos (`t0`, `t1`) e um com a desquantização exata em F16 (`e0`, `e1`), que mede a qualidade no mesmo caminho F16 dos k-quants.

In [ ]:
r = !{LIVRO} --gguf {G}/Q4_K_M_imat.gguf --modo bloco --bits 4 5 --formato --json /content/livro.json --out /content/e0.gguf /content/e1.gguf --out-tipo {G}/LIVRO0.gguf {G}/LIVRO1.gguf 2>&1
print('\n'.join(l for l in r if l.startswith('nível') or 'Error' in l or 'Traceback' in l or 'rror' in l))
res = json.load(open('/content/livro.json'))
for n, nv in enumerate(res['niveis']):
    for nome in (f'livro_N{n}', f'LIVRO{n}_gpu', f'LIVRO{n}_ub8', f'LIVRO{n}_conf_cpu', f'LIVRO{n}_conf_gpu', f'LIVRO{n}_conf_ub8'):
        bpw[nome] = nv['bits_por_peso']
    erro[f'livro_N{n}'] = nv['erro_rel']
!ls -la /content/e*.gguf {G}/LIVRO*.gguf
# o formato real desquantizado pelo ggml tem de dar, peso a peso, a emulação em F16
!cc -O2 /content/strata/tests/kernel/confere_livro_gguf.c -I llama.cpp/ggml/include -L {LIB} -lggml -lggml-base -lm -Wl,-rpath,{LIB} -o confere_livro_gguf
for n in (0, 1):
    out = !./confere_livro_gguf {G}/LIVRO{n}.gguf /content/e{n}.gguf
    print(f'nível {n}:', '\n'.join(out))
    assert 'diferentes depois do arredondamento F16: 0;' in '\n'.join(out), 'o tipo novo não dá a emulação: avise na thread'

Qualidade e conferência do kernel. Para cada nível: a emulação em F16 (quem decide a qualidade), o tipo novo na GPU com lote normal (desquantização + cuBLAS, ativações em F16, o mesmo caminho da emulação) e com lote de 8 (produto escalar, o caminho da geração, ativações em q8_1). Nos 5 trechos de conferência, CPU contra GPU com lote de 8: os dois arredondam as ativações para 8 bits (a CPU em Q8_K, a GPU em q8_1), então é a comparação de igual para igual; a CPU contra a GPU com lote normal mistura o efeito desse arredondamento (no Llama-3.2-1B, +5% a +19% de KL nos tipos do livro e nos k-quants). O IQ4_XS e o Q5_K_S passam pelas mesmas medidas, de controle.

In [ ]:
for n in (0, 1):
    kl_de(f'/content/e{n}.gguf', f'livro_N{n}')
    !rm -f /content/e{n}.gguf
    kl_de(f'{G}/LIVRO{n}.gguf', f'LIVRO{n}_gpu')
    kl_de(f'{G}/LIVRO{n}.gguf', f'LIVRO{n}_ub8', extra=GPU_ARGS + ' -ub 8')
    kl_de(f'{G}/LIVRO{n}.gguf', f'LIVRO{n}_conf_gpu', base='/content/kld_conf.bin')
    kl_de(f'{G}/LIVRO{n}.gguf', f'LIVRO{n}_conf_ub8', base='/content/kld_conf.bin', extra=GPU_ARGS + ' -ub 8')
    kl_de(f'{G}/LIVRO{n}.gguf', f'LIVRO{n}_conf_cpu', base='/content/kld_conf.bin', extra=CPU_ARGS)
# os mesmos caminhos nos tipos do llama.cpp, de controle
for q in ['IQ4_XS_imat', 'Q5_K_S_imat']:
    kl_de(f'{G}/{q}.gguf', q + '_ub8', extra=GPU_ARGS + ' -ub 8')
    kl_de(f'{G}/{q}.gguf', q + '_conf_ub8', base='/content/kld_conf.bin', extra=GPU_ARGS + ' -ub 8')
    kl_de(f'{G}/{q}.gguf', q + '_conf_cpu', base='/content/kld_conf.bin', extra=CPU_ARGS)

Velocidade na GPU: o nível 0 contra o IQ4_XS (o mesmo bloco, com a tabela fixa) e o nível 1 contra o Q5_K_S, mais o Q4_K_M de referência. Os modelos são alternados em ordem aleatória a cada rodada; vale a mediana.

In [ ]:
import random, statistics
VEL = {'LIVRO0': f'{G}/LIVRO0.gguf', 'LIVRO1': f'{G}/LIVRO1.gguf', 'IQ4_XS_imat': f'{G}/IQ4_XS_imat.gguf',
       'Q5_K_S_imat': f'{G}/Q5_K_S_imat.gguf', 'Q4_K_M_imat': f'{G}/Q4_K_M_imat.gguf'}
amostras = {}
for r_ in range(RODADAS):
    ordem = list(VEL)
    random.shuffle(ordem)
    for m in ordem:
        for k, v in bench(VEL[m]).items():
            amostras.setdefault((m, k), []).append(v)
    print(f'rodada {r_ + 1} de {RODADAS} pronta')
for (m, k), v in amostras.items():
    vel.setdefault(m, {})[k] = statistics.median(v)
for m, v in vel.items():
    print(m, {k: round(x, 1) for k, x in v.items()})

A curva: k-quants, IQ4_XS e Q8_0 com as matrizes das camadas desquantizadas para F16 (~16 GB cada), o mesmo caminho da emulação do livro. Os GGUFs quantizados vão sendo apagados.

In [ ]:
for q in ['Q3_K_M_imat', 'Q4_K_S_imat', 'Q4_K_M_imat', 'IQ4_XS_imat', 'Q5_K_S_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q6_K_Q8mix_imat', 'Q8_0']:
    r = !{LIVRO} --gguf {G}/{q}.gguf --modo copia --json /content/F16_{q}.json --out /content/e0.gguf 2>&1
    nv = json.load(open(f'/content/F16_{q}.json'))['niveis'][0]
    bpw['F16_' + q], erro['F16_' + q] = nv['bits_por_peso'], nv['erro_rel']
    kl_de('/content/e0.gguf', 'F16_' + q)
    !rm -f /content/e0.gguf {G}/{q}.gguf
!df -h /content | tail -1

**A régua.** Para cada nível: bits por peso nas matrizes das camadas (com escalas e tabelas); a KL dos k-quants com imatrix em F16 interpolada (log) no mesmo tamanho; e os bits a mais que o candidato gasta além da fronteira de mesma KL (abaixo de 0 = o livro ganha), com e sem o IQ4_XS na fronteira. O veredito usa a fronteira com o IQ4_XS, como no Llama-3.1-8B.

In [ ]:
import math
from IPython.display import Markdown, display
KQ = ['Q3_K_M_imat', 'Q4_K_S_imat', 'Q4_K_M_imat', 'Q5_K_S_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q6_K_Q8mix_imat', 'Q8_0']

def pontos(com_iq4=False):
    qs = KQ + (['IQ4_XS_imat'] if com_iq4 else [])
    p = sorted((bpw['F16_' + q], kld['F16_' + q][0]) for q in qs if 'F16_' + q in kld)
    f = []                                  # só a fronteira: cada ponto com KL menor que todos os de menos bits
    for s, k in p:
        if not f or k < f[-1][1]:
            f.append((s, k))
    return f

def curva(s, pts):
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if s <= s2 or (s2, k2) == pts[-1]:
            if s < pts[0][0]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return k1 * math.exp(math.log(k2 / k1) * (s - s1) / (s2 - s1))

def bits_igual(k, pts):
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if k >= k2 or (s2, k2) == pts[-1]:
            if k > pts[0][1]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return s1 + (s2 - s1) * math.log(k / k1) / math.log(k2 / k1)

fmt = lambda x, f='.3f': 'n/d' if x is None else format(x, f)
T = [f'# STRATA, livro de códigos aninhado no formato real ({MODELO_HF})', '',
     f'GPU: {GPU}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. '
     'Bits por peso nas matrizes das camadas.', '',
     '| modelo | bits/peso | KL (×1000) | mesmo 1º token | erro ponderado |', '|---|---:|---:|---:|---:|']
for m in sorted([m for m in kld if '_conf' not in m and '_ub8' not in m], key=lambda m: bpw.get(m) or 99):
    v = kld[m]
    T.append(f'| {m} | {fmt(bpw.get(m))} | {1000 * v[0]:.3f} ± {1000 * v[1]:.3f} | {v[2]:.2f}% | {fmt(100 * erro[m] if m in erro else None, ".4f")}% |')
P, PI = pontos(), pontos(True)
T += ['', '## Régua de qualidade (curva = k-quants com imatrix em F16)', '',
      '| nível | bits/peso | KL ×1000 | curva no mesmo tamanho | razão | bits a mais | bits a mais (curva com IQ4_XS) |',
      '|---|---:|---:|---:|---:|---:|---:|']
extra, extra_iq = {}, {}
for n in (0, 1):
    m = f'livro_N{n}'
    if m not in kld:
        continue
    s, k = bpw[m], kld[m][0]
    extra[m], extra_iq[m] = s - bits_igual(k, P), s - bits_igual(k, PI)
    T.append(f'| {n} | {s:.3f} | {1000 * k:.3f} | {1000 * curva(s, P):.3f} | {k / curva(s, P):.3f} | {extra[m]:+.3f} | {extra_iq[m]:+.3f} |')
e = [extra_iq.get(f'livro_N{n}') for n in (0, 1)]
if None in e:
    v = 'incompleto'
elif max(e) <= 0.05:
    v = 'sucesso'
elif max(e) <= 0.10:
    v = 'zona cinza'
else:
    v = 'fracasso'
T += ['', f'**Veredito de qualidade (régua de docs/09): {v}**' + ('' if None in e else f' (níveis 0/1: {e[0]:+.3f} / {e[1]:+.3f} bit/peso)'), '',
      'Llama-3.1-8B (livro em float, 2026-10-23): +0,004 / +0,006.', '',
      '**Dominância** (menos bits e menos KL ao mesmo tempo, sem interpolação):', '']
for n in (0, 1):
    m = f'livro_N{n}'
    if m in kld:
        dom = [q.replace('F16_', '') for q in kld if q.startswith('F16_') and bpw[m] <= bpw[q] and kld[m][0] <= kld[q][0]]
        T.append(f'- nível {n}: ' + (', '.join(dom) if dom else 'nenhum'))

T += ['', '## Conferência do kernel', '',
      '| modelo | caminho F16 | GPU, lote normal | diferença / erro padrão | GPU, lote 8 | CPU (5 trechos) | GPU lote 8 (5 trechos) | CPU − GPU lote 8 / erro padrão |',
      '|---|---:|---:|---:|---:|---:|---:|---:|']
conf_ok = True
for n in (0, 1):
    g = lambda s: kld.get(f'LIVRO{n}_{s}')
    em, gp, u8, cc, cu = kld.get(f'livro_N{n}'), g('gpu'), g('ub8'), g('conf_cpu'), g('conf_ub8')
    if None in (em, gp, cc, cu):
        conf_ok = False
        T.append(f'| nível {n} | incompleto | | | | | | |')
        continue
    z1 = (gp[0] - em[0]) / em[1]
    z2 = (cc[0] - cu[0]) / cu[1]
    conf_ok &= abs(z1) <= 1 and abs(z2) <= 1
    T.append(f'| nível {n} | {1000 * em[0]:.3f} | {1000 * gp[0]:.3f} | {z1:+.2f} | {fmt(u8 and 1000 * u8[0])} | {1000 * cc[0]:.3f} | {1000 * cu[0]:.3f} | {z2:+.2f} |')
for q in ['IQ4_XS_imat', 'Q5_K_S_imat']:
    em, u8, cc, cu = kld.get('F16_' + q), kld.get(q + '_ub8'), kld.get(q + '_conf_cpu'), kld.get(q + '_conf_ub8')
    if None in (em, u8, cc, cu):
        continue
    T.append(f'| {q} (controle) | {1000 * em[0]:.3f} | | | {1000 * u8[0]:.3f} | {1000 * cc[0]:.3f} | {1000 * cu[0]:.3f} | {(cc[0] - cu[0]) / cu[1]:+.2f} |')
T += ['', f'**Conferência: {"passou" if conf_ok else "NÃO passou"}** (as duas diferenças dos níveis a no máximo 1 erro padrão). KL ×1000. '
      'A coluna "GPU, lote 8" mostra o efeito de arredondar as ativações para 8 bits na geração; ele deve ser do mesmo tamanho nos tipos do livro e nos de controle.']

T += ['', '## Velocidade na GPU (tok/s, mediana de rodadas alternadas)', '', '| modelo | geração (tg128) | prompt (pp512) |', '|---|---:|---:|']
for m, v_ in vel.items():
    T.append(f'| {m} | {v_.get("tg", float("nan")):.1f} | {v_.get("pp", float("nan")):.0f} |')
def razao(a, b, k):
    return vel[a][k] / vel[b][k] if a in vel and b in vel and k in vel[a] and k in vel[b] else None
r0, r1 = razao('LIVRO0', 'IQ4_XS_imat', 'tg'), razao('LIVRO1', 'Q5_K_S_imat', 'tg')
T += ['', f'- geração, nível 0 / IQ4_XS: {fmt(r0, ".3f")} (régua ≥ 0,95: {"passou" if r0 and r0 >= 0.95 else "NÃO passou"})',
      f'- geração, nível 1 / Q5_K_S: {fmt(r1, ".3f")} (régua ≥ 0,95: {"passou" if r1 and r1 >= 0.95 else "NÃO passou"})',
      f'- prompt, nível 0 / IQ4_XS: {fmt(razao("LIVRO0", "IQ4_XS_imat", "pp"), ".3f")}; nível 1 / Q5_K_S: {fmt(razao("LIVRO1", "Q5_K_S_imat", "pp"), ".3f")} '
      '(fora da régua: os tipos do livro leem o prompt por desquantização + cuBLAS; os do llama.cpp, pelo MMQ)']

os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_qwen3_livro_kernel.md', 'w').write('\n'.join(T) + '\n')
json.dump({'modelo': MODELO_HF, 'texto': TEXTO, 'gpu': GPU, 'kld_chunks': KLD_CHUNKS, 'conf_chunks': CONF_CHUNKS,
           'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld, 'bits_por_peso': bpw, 'erro_rel': erro, 'vel': vel,
           'vel_amostras': {f'{m}|{k}': v_ for (m, k), v_ in amostras.items()},
           'bits_a_mais': extra, 'bits_a_mais_com_iq4': extra_iq, 'veredito': v, 'conferencia_ok': conf_ok},
          open('/content/res/strata_qwen3_livro_kernel.json', 'w'), indent=1)
display(Markdown('\n'.join(T)))

Mande os dois arquivos na thread (ou cole a tabela).

In [ ]:
from google.colab import files
files.download('/content/res/strata_qwen3_livro_kernel.md')
files.download('/content/res/strata_qwen3_livro_kernel.json')